# RAUSNet — improve the global model (one click)

**How to use:** `Runtime > Run all` (or **Ctrl+F9**), then allow access to Google Drive. Nothing else.
The notebook opens with a free T4 GPU already selected and releases it at the end.

What happens:
1. The shared workspace folder on Drive (`RAUSNet/`) is checked for **new labelled images**
   (clinician corrections sent from the desktop app, new batches). Fewer than `min_new_images`
   (default 20) → it stops in about a minute and frees the GPU.
2. Otherwise the current global model is fine-tuned on everything, compared with it on the frozen
   test split (paired bootstrap), calibrated and exported to ONNX.
3. If it is better or not significantly worse it becomes the new global model: every desktop app
   that uses this workspace gets it at its next start. The result is a single line in
   `RAUSNet/improve_log.jsonl`.

An empty workspace is set up automatically (synthetic seed data + first model, ~15 min).

**Rules:** free tier only; de-identified data only (`rausnet-deid`), and clinical images on Drive only
if the ethics approval allows it; never save this notebook with outputs to the repository. If the
session disconnects nothing is published; run it again (a stale lock is cleared after 6 h).

## Settings (optional)

In [ ]:
WORKSPACE = "/content/drive/MyDrive/RAUSNet"   # shared folder; for "Shared with me" add a shortcut in My Drive
BRANCH = "PIA_devl"
MIN_NEW = None        # None = value in configs/retrain/default.yaml (improve.min_new_images)
AUTO_STOP = True      # release the GPU at the end

## Setup: GPU, Drive, code

In [ ]:
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU works, slower)")
from google.colab import drive
drive.mount("/content/drive")
REPO = "/content/rausnet"
if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only -q
else:
    !git clone -q --depth 1 -b {BRANCH} https://github.com/JB-GB/AI_for_Biomedical_Engineers.git {REPO}
!pip install -q -e "{REPO}[train,serve]" onnxscript
os.chdir(REPO)

## Improve the global model

In [ ]:
min_new = "" if MIN_NEW is None else f"--min-new {MIN_NEW}"
# a failure here does not stop "Run all": the next cells still report and release the GPU
!python -m rausnet.improve run --workspace "{WORKSPACE}" --staging /content/stage {min_new}

## Status (numbers only)

In [ ]:
import json, subprocess
s = json.loads(subprocess.run(["python", "-m", "rausnet.improve", "status", "--workspace", WORKSPACE],
                              capture_output=True, text=True, check=True).stdout)
cur = s["current_model"] or {}
print("global model:", cur.get("version"), "| test QWK:", (cur.get("test_metrics") or {}).get("qwk"))
print("waiting images:", s["pending_images"], "| incomplete batches:", len(s["incomplete_batches"]))
print("last run:", s["history"][-1] if s["history"] else None)

## Release the GPU

In [ ]:
if AUTO_STOP:
    from google.colab import runtime
    runtime.unassign()